In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
df = spark.read.table("workspace.bronze.erp_loc_a101")
display(df)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

In [0]:
df.select(F.length(F.col("CID"))).distinct().show()

In [0]:
df = df.withColumn("CID", F.substring(F.col("CID"), F.length(F.col("CID")) - 4, F.length(F.col("CID"))))  

In [0]:
df.select("CNTRY").distinct().show()

In [0]:
df = df.withColumn("CNTRY", 
                   F.when(F.col("CNTRY").isin("US", "USA"), "United States")
                   .when(F.col("CNTRY") == "DE", "Germany")
                   .when(F.col("CNTRY").isin("NULL", ""), "Unknown")
                   .otherwise(F.col("CNTRY"))
                  )

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "CNTRY": "country"
}

df = df.withColumnsRenamed(RENAME_MAP)


In [0]:
display(df)

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_locations")

In [0]:
%sql
SELECT * 
FROM workspace.silver.erp_locations
LIMIT 10